In [2]:
import pandas as pd

DATA_PATH = '../data/ready_to_use'

patients = pd.read_csv(f'{DATA_PATH}/patients_cleaned.csv')

In [3]:
patients.head()

,patient_id,sex_at_birth,race_ethnicity_group,preferred_language,interpreter_needed_flag,zip_area_id,payer_primary_id,pcp_provider_id,chronic_condition_count,sdoh_risk_score
0,RBP000001,Female,Black non-Hispanic,Spanish,1,ZA003,PAY04,RBPV0019,2,45.2
1,RBP000002,Male,Hispanic/Latino,Spanish,1,ZA028,PAY01,RBPV0027,0,41.7
2,RBP000003,Female,White non-Hispanic,English,0,ZA015,PAY01,RBPV0022,4,57.1
3,RBP000004,Male,Hispanic/Latino,English,0,ZA008,PAY03,RBPV0005,2,33.7
4,RBP000005,Male,White non-Hispanic,English,0,ZA007,PAY01,RBPV0001,3,38.4


In [4]:
patients.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient_id               12000 non-null  str    
 1   sex_at_birth             12000 non-null  str    
 2   race_ethnicity_group     12000 non-null  str    
 3   preferred_language       12000 non-null  str    
 4   interpreter_needed_flag  12000 non-null  int64  
 5   zip_area_id              12000 non-null  str    
 6   payer_primary_id         12000 non-null  str    
 7   pcp_provider_id          12000 non-null  str    
 8   chronic_condition_count  12000 non-null  int64  
 9   sdoh_risk_score          12000 non-null  float64
dtypes: float64(1), int64(2), str(7)
memory usage: 937.6 KB


In [5]:
patients.isnull().sum()

patient_id                 0
sex_at_birth               0
race_ethnicity_group       0
preferred_language         0
interpreter_needed_flag    0
zip_area_id                0
payer_primary_id           0
pcp_provider_id            0
chronic_condition_count    0
sdoh_risk_score            0
dtype: int64

In [6]:
patients.duplicated().sum()

np.int64(0)

## Cleaning

### handle 'not recorded' value of the 'preferred_language' feature

In [7]:
patients['preferred_language'].unique()

<StringArray>
['Spanish', 'English', 'Arabic', 'Mandarin', 'Not recorded', 'Nepali']
Length: 6, dtype: str

In [8]:
patients.iloc[patients['preferred_language'] == 'Not recorded']['preferred_language'].count()

np.int64(351)

In [9]:
lang_mode = patients['preferred_language'].mode()[0]
patients['preferred_language'] = patients['preferred_language'].replace(['Not recorded'], lang_mode)

In [10]:
patients['preferred_language'].unique()

<StringArray>
['Spanish', 'English', 'Arabic', 'Mandarin', 'Nepali']
Length: 5, dtype: str

### handle 'not recorded' value of the 'sex_at_birth' feature

In [11]:
patients['sex_at_birth'].unique()

<StringArray>
['Female', 'Male', 'Not recorded', 'Intersex']
Length: 4, dtype: str

In [12]:
print(f"#not recorded values: {patients.iloc[patients['sex_at_birth'] == 'Not recorded']['sex_at_birth'].count()}")
print(f"#intersex values: {patients.iloc[patients['sex_at_birth'] == 'Intersex']['sex_at_birth'].count()}")

#not recorded values: 110
#intersex values: 63


In [13]:
gender_mode = patients['sex_at_birth'].mode()[0]
patients['sex_at_birth'] = patients['sex_at_birth'].replace(['Not recorded','Intersex'], gender_mode)

In [14]:
patients['sex_at_birth'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

### handle 'not recorded' value of the 'race_ethnicity_group' feature

In [15]:
patients['race_ethnicity_group'].unique()

<StringArray>
['Black non-Hispanic',    'Hispanic/Latino', 'White non-Hispanic',
  'Multiracial/Other',       'Not recorded',              'Asian']
Length: 6, dtype: str

In [16]:
print(f"#not recorded values: {patients.iloc[patients['race_ethnicity_group'] == 'Not recorded']['race_ethnicity_group'].count()}")

#not recorded values: 226


In [17]:
patients['race_ethnicity_group'] = patients['race_ethnicity_group'].str.replace(' non-Hispanic', '')

In [18]:
race_mode = patients['race_ethnicity_group'].mode()[0]
patients['race_ethnicity_group'] = patients['race_ethnicity_group'].replace(['Not recorded'], race_mode)

In [19]:
patients['race_ethnicity_group'].unique()

<StringArray>
['Black', 'Hispanic/Latino', 'White', 'Multiracial/Other', 'Asian']
Length: 5, dtype: str

### handling 'sdoh_risk_score' feature outliers

In [20]:
Q1 = patients['sdoh_risk_score'].quantile(0.25)
Q3 = patients['sdoh_risk_score'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = max(0, Q1 - 1.5 * IQR)
upper_bound = Q3 + 1.5 * IQR

print(f"sdoh_risk_score lower bound: {lower_bound}, upper bound: {upper_bound}")
print(f"values < lower = {patients['sdoh_risk_score'][patients['sdoh_risk_score'] < lower_bound].count()}")
print(f"values > upper = {patients['sdoh_risk_score'][patients['sdoh_risk_score'] > upper_bound].count()}")

sdoh_risk_score lower bound: 6.899999999999999, upper bound: 90.9
values < lower = 16
values > upper = 29


In [21]:
patients = patients[(patients['sdoh_risk_score'] >= lower_bound) & (patients['sdoh_risk_score'] <= upper_bound)]

In [22]:
print(f"values < lower = {patients['sdoh_risk_score'][patients['sdoh_risk_score'] < lower_bound].count()}")
print(f"values > upper = {patients['sdoh_risk_score'][patients['sdoh_risk_score'] > upper_bound].count()}")

values < lower = 0
values > upper = 0


### handling 'chronic_condition_count' feature outliers

In [23]:
Q1 = patients['chronic_condition_count'].quantile(0.25)
Q3 = patients['chronic_condition_count'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = max(0, Q1 - 1.5 * IQR)
upper_bound = Q3 + 1.5 * IQR

print(f"chronic_condition_count lower bound: {lower_bound}, upper bound: {upper_bound}")
print(f"values < lower = {patients['chronic_condition_count'][patients['chronic_condition_count'] < lower_bound].count()}")
print(f"values > upper = {patients['chronic_condition_count'][patients['chronic_condition_count'] > upper_bound].count()}")

chronic_condition_count lower bound: 0, upper bound: 7.5
values < lower = 0
values > upper = 25


In [24]:
patients = patients[(patients['chronic_condition_count'] >= lower_bound) & (patients['chronic_condition_count'] <= upper_bound)]

In [25]:
print(f"values < lower = {patients['chronic_condition_count'][patients['chronic_condition_count'] < lower_bound].count()}")
print(f"values > upper = {patients['chronic_condition_count'][patients['chronic_condition_count'] > upper_bound].count()}")

values < lower = 0
values > upper = 0


In [28]:
patients.describe()

,interpreter_needed_flag,chronic_condition_count,sdoh_risk_score
count,11930.000000,11930.000000,11930.000000
mean,0.126236,1.671417,48.973822
std,0.332129,1.475140,15.003890
min,0.000000,0.000000,7.000000
25%,0.000000,0.000000,38.400000
50%,0.000000,1.000000,48.900000
75%,0.000000,3.000000,59.400000
max,1.000000,7.000000,90.900000


In [27]:
path = '../data/clean'

patients.to_csv(f'{path}/patients_clean.csv', index=False)